In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, max as spark_max, min as spark_min, round as spark_round, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

spark = SparkSession.builder.appName("FlightInsights").getOrCreate()

HDFS_PATH = "hdfs://localhost:9000/flights/flights/"

raw = spark.read.json(HDFS_PATH)
print("Raw files read. Schema:")
raw.printSchema()

df = raw.select(
    col("payload.flight_id").alias("flight_id"),
    col("payload.fleet").alias("fleet"),
    col("payload.great_circle_distance_nm").alias("distance_nm"),
    col("payload.gross_weight_at_liftoff_kg").alias("weight_kg"),
    col("payload.max_tailwind_during_takeoff_kt").alias("tailwind_kt"),
    col("payload.generated_at").alias("generated_at"),
)
df = df.dropDuplicates(["flight_id"])

print(f"\nTotal unique flights loaded from HDFS: {df.count()}")
df.show(5, truncate=False)

print("\n" + "=" * 70)
print("INSIGHT 1: Average distance and weight per fleet")
print("=" * 70)
insight1 = df.groupBy("fleet").agg(
    count("*").alias("n_flights"),
    spark_round(avg("distance_nm"), 1).alias("avg_distance_nm"),
    spark_round(avg("weight_kg"), 0).alias("avg_weight_kg"),
)
insight1.orderBy("fleet").show()

print("=" * 70)
print("INSIGHT 2: Distance range per fleet (min - max)")
print("=" * 70)
insight2 = df.groupBy("fleet").agg(
    spark_round(spark_min("distance_nm"), 0).alias("min_distance_nm"),
    spark_round(spark_max("distance_nm"), 0).alias("max_distance_nm"),
    spark_round(spark_max("distance_nm") - spark_min("distance_nm"), 0).alias("range_nm"),
)
insight2.orderBy("fleet").show()

print("=" * 70)
print("INSIGHT 3: Headwind vs tailwind flight counts per fleet")
print("=" * 70)
df_wind = df.withColumn(
    "wind_condition",
    when(col("tailwind_kt") > 0, "tailwind")
    .when(col("tailwind_kt") < 0, "headwind")
    .otherwise("calm")
)
insight3 = df_wind.groupBy("fleet", "wind_condition").count()
insight3.orderBy("fleet", "wind_condition").show()

print("=" * 70)
print("INSIGHT 4: Heaviest recorded flight per fleet")
print("=" * 70)
w = Window.partitionBy("fleet").orderBy(col("weight_kg").desc())
heaviest = (
    df.withColumn("rank", row_number().over(w))
      .filter(col("rank") == 1)
      .select("fleet", "flight_id", "distance_nm", "weight_kg")
)
heaviest.show(truncate=False)

print("\nDone.")

Py4JJavaError: An error occurred while calling o39.json.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 9070) (DESKTOP-Q9RBV34 executor driver): org.apache.spark.SparkException: Encountered error while reading file hdfs://localhost:9000/flights/flights/partition=2/flights+2+0000000000+0000000002.json. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:877)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:307)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:125)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:491)
	at scala.collection.Iterator.isEmpty(Iterator.scala:387)
	at scala.collection.Iterator.isEmpty$(Iterator.scala:387)
	at scala.

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, max as spark_max, min as spark_min, round as spark_round, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

spark = SparkSession.builder.appName("FlightInsights").getOrCreate()
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

raw = spark.read.json("hdfs://localhost:9000/flights/flights/partition=0/")

df = raw.select(
    col("payload.flight_id").alias("flight_id"),
    col("payload.fleet").alias("fleet"),
    col("payload.great_circle_distance_nm").alias("distance_nm"),
    col("payload.gross_weight_at_liftoff_kg").alias("weight_kg"),
    col("payload.max_tailwind_during_takeoff_kt").alias("tailwind_kt"),
    col("payload.generated_at").alias("generated_at"),
)
df = df.dropDuplicates(["flight_id"])

print(f"Total unique flights loaded: {df.count()}")

Py4JJavaError: An error occurred while calling o84.json.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 17.0 failed 1 times, most recent failure: Lost task 1.0 in stage 17.0 (TID 19021) (DESKTOP-Q9RBV34 executor driver): org.apache.spark.SparkException: Encountered error while reading file hdfs://localhost:9000/flights/flights/partition=0/flights+0+0000000449+0000000449.json. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:877)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:307)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:125)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:491)
	at scala.collection.Iterator.isEmpty(Iterator.scala:387)
	at scala.collection.Iterator.isEmpty$(Iterator.scala:387)
	at sca

In [11]:
from pyspark.sql.functions import col, avg, count, max as spark_max, min as spark_min, round as spark_round, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("\n" + "=" * 70)
print("INSIGHT 1: Average distance and weight per fleet")
print("=" * 70)
insight1 = df.groupBy("fleet").agg(
    count("*").alias("n_flights"),
    spark_round(avg("distance_nm"), 1).alias("avg_distance_nm"),
    spark_round(avg("weight_kg"), 0).alias("avg_weight_kg"),
)
insight1.orderBy("fleet").show()

print("=" * 70)
print("INSIGHT 2: Distance range per fleet (min - max)")
print("=" * 70)
insight2 = df.groupBy("fleet").agg(
    spark_round(spark_min("distance_nm"), 0).alias("min_distance_nm"),
    spark_round(spark_max("distance_nm"), 0).alias("max_distance_nm"),
    spark_round(spark_max("distance_nm") - spark_min("distance_nm"), 0).alias("range_nm"),
)
insight2.orderBy("fleet").show()

print("=" * 70)
print("INSIGHT 3: Headwind vs tailwind flight counts per fleet")
print("=" * 70)
df_wind = df.withColumn(
    "wind_condition",
    when(col("tailwind_kt") > 0, "tailwind")
    .when(col("tailwind_kt") < 0, "headwind")
    .otherwise("calm")
)
insight3 = df_wind.groupBy("fleet", "wind_condition").count()
insight3.orderBy("fleet", "wind_condition").show()

print("=" * 70)
print("INSIGHT 4: Heaviest recorded flight per fleet")
print("=" * 70)
w = Window.partitionBy("fleet").orderBy(col("weight_kg").desc())
heaviest = (
    df.withColumn("rank", row_number().over(w))
      .filter(col("rank") == 1)
      .select("fleet", "flight_id", "distance_nm", "weight_kg")
)
heaviest.show(truncate=False)


INSIGHT 1: Average distance and weight per fleet


NameError: name 'df' is not defined